# nb25 - Inferring the tSZ hydrostatic bias B (arnaudB1 selection, ell>100)

**Updated** to the corrected pipeline. The masked tSZ bandpowers are built from the
**A10 GNFW (B=1) SNR selection** (q from `halo_catalogue_..._y0q_arnaudB1.csv`,
positions from yang26rot), NaMaster + monopole-subtracted, in
`data/bandpowers_arnaudB1/`. The covariance is the theory Gaussian (Knox) + 1-halo
trispectrum, `M = C_G + T/(4 pi f_sky)`, in `data/theory_cov_arnaudB1/`.

We fix the D3A cosmology (A_s=2.099e-9), the A10 GNFW shape (hmfast defaults) and
**sigma_lnY = 0.173**, and sample **only the signal bias B** on the **last 9
bandpowers (ell>100)** for the full-sky and q>{5,10,20,50} cuts. Chains:
`chains/mcmc_arnaudB1_fitB_ellgt100/`. This notebook shows the B posteriors and the
best-fit tSZ power spectra against the bandpowers.

In [1]:
import os, sys
os.environ["HMFAST_COBAYA_USE_GPU"]="1"; os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
REPO="/scratch/scratch-lxu/flamingo_repo"
sys.path.insert(0, os.path.join(REPO,"cobaya","theory"))
import numpy as np, matplotlib.pyplot as plt, matplotlib.colors as mcolors, jax.numpy as jnp

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

from getdist import loadMCSamples, plots
from hmfast_tsz_iter_cprofile import _bin_to_18, _ELL_MIN, _ELL_MAX
from hmfast.cosmology import Cosmology
from hmfast.halos import HaloModel
from hmfast.halos.mass_definition import MassDefinition
from hmfast.halos.profiles import GNFWPressureProfile
from hmfast.tracers import tSZTracer
from hmfast.tracers.tsz_completeness import build_snr_grid, conditional_An_undetected, load_sigma_y0_curve


## Configuration (matches the chains and the covariance)

In [2]:
CHAIN_DIR=os.path.join(REPO,"chains","mcmc_arnaudB1_fitB_ellgt100")
BP_DIR=os.path.join(REPO,"data","bandpowers_arnaudB1")
CV_DIR=os.path.join(REPO,"data","theory_cov_arnaudB1")
KEEP=9                       # last 9 of 18 bins -> ell>100
LN1E10_AS=3.044046           # A_s=2.099e-9 (D3A)
SIGMA_LNY=0.173              # fixed
A_SZ_SR, ALPHA_SR, B_SR = -4.287396, 1.12, 1.0   # A10-GNFW completeness (== cov)
D3A=dict(H0=68.1, omega_cdm=0.118729, omega_b=0.022539, n_s=0.967, tau_reio=0.0544)
CUTS=[("fullsky","Full sky",1.0e9),("qgt5","q > 5",5.0),("qgt10","q > 10",10.0),
      ("qgt20","q > 20",20.0),("qgt50","q > 50",50.0)]
# f_sky_eff used in theory_cov_arnaudB1 (matches compute_arnaudB1_full_cov.py)
FSKY=dict(fullsky=1.0, qgt5=0.7690275812769604, qgt10=0.9008929347361354,
          qgt20=0.9755782103556229, qgt50=0.9990961741880922)
colors=[mcolors.to_hex(c) for c in plt.cm.plasma(np.linspace(0,0.85,len(CUTS)))]
OUTDIR=os.path.join(REPO,"figures","nb25_arnaudB1_fitB_ellgt100"); os.makedirs(OUTDIR,exist_ok=True)

## B posteriors

In [3]:
samples,labels,bestfit=[],[],{}
for tag,lab,_ in CUTS:
    s=loadMCSamples(os.path.join(CHAIN_DIR,f"yy_{tag}_fitB"),settings={"ignore_rows":0.3})
    s.root=lab; samples.append(s); labels.append(lab)
    B=np.asarray(s.getParams().B); bestfit[tag]=float(B[int(np.argmin(s.loglikes))])
    print(f"{lab:8s}  B = {s.mean('B'):.4f} +/- {s.std('B'):.4f}  (MAP {bestfit[tag]:.4f})")

g=plots.get_subplot_plotter(width_inch=7.0); g.settings.legend_fontsize=11
g.plot_1d(samples,"B",colors=colors); g.add_legend(labels,legend_loc="upper right",colored_text=True)
g.subplots[0,0].set_title(r"Posterior of signal bias $B$ (arnaudB1, $\ell>100$, $\sigma_{\ln Y}=0.173$)",fontsize=10)
g.export(os.path.join(OUTDIR,"posterior_B_arnaudB1_ellgt100.pdf")); g.export(os.path.join(OUTDIR,"posterior_B_arnaudB1_ellgt100.png"), dpi=300)
plt.show()


Full sky  B = 1.1622 +/- 0.0108  (MAP 1.1619)
q > 5     B = 1.0656 +/- 0.0022  (MAP 1.0656)
q > 10    B = 1.1012 +/- 0.0033  (MAP 1.1006)
q > 20    B = 1.1249 +/- 0.0046  (MAP 1.1244)
q > 50    B = 1.1517 +/- 0.0075  (MAP 1.1506)


## hmfast best-fit tSZ power spectrum vs bandpowers

Theory matches the chains: A10 GNFW hmfast defaults, D3A (A_s=2.099e-9), masking
completeness from the A10 GNFW y0 (A_SZ=-4.2874, alpha=1.12, B_SR=1.0), sigma_lnY=0.173,
B at its best fit. Compared to the new bandpowers (ell>100) with the diagonal of the
new full covariance (Gaussian + trispectrum).

In [4]:
m=jnp.geomspace(1e10,10**15.5,64); z=jnp.geomspace(0.005,3.0,96)
ell_int=jnp.geomspace(float(_ELL_MIN[0]),float(_ELL_MAX[-1]),50); ell_np=np.asarray(ell_int)
cosmo=Cosmology(emulator_set="lcdm:v1").update(H0=D3A["H0"],omega_cdm=D3A["omega_cdm"],
        omega_b=D3A["omega_b"],ln1e10A_s=LN1E10_AS,n_s=D3A["n_s"],tau_reio=D3A["tau_reio"])
hm=HaloModel(cosmology=cosmo,mass_definition=MassDefinition(500,"critical"),convert_masses=True)
prof_seed=GNFWPressureProfile(B=1.0)        # hmfast A10 defaults
tsz_seed=tSZTracer(profile=prof_seed)
coeff,_=load_sigma_y0_curve()
snr=build_snr_grid(hm,m,z,A_SZ_SR,ALPHA_SR,B_SR,coeff=coeff)   # completeness grid (cut-independent)

def predict_dl(q_cat,B):
    tsz=tsz_seed.update(profile=prof_seed.update(B=B))
    if q_cat>=1e8:   # full sky: scatter boost only
        mask1=conditional_An_undetected(snr,sigma_lnY=SIGMA_LNY,q_cat=q_cat,n_power=2)
        mask2=conditional_An_undetected(snr,sigma_lnY=SIGMA_LNY,q_cat=q_cat,n_power=1)
    else:
        mask1=conditional_An_undetected(snr,sigma_lnY=SIGMA_LNY,q_cat=q_cat,n_power=2)
        mask2=conditional_An_undetected(snr,sigma_lnY=SIGMA_LNY,q_cat=q_cat,n_power=1)
    cl1=hm.cl_1h_masked(tsz,None,ell_int,m,z,mask1)
    cl2=hm.cl_2h_masked(tsz,None,ell_int,m,z,mask2)
    return _bin_to_18(ell_np,np.asarray(cl1))+_bin_to_18(ell_np,np.asarray(cl2))

from scipy.stats import chi2 as _chi2
delta_ell_all = (_ELL_MAX - _ELL_MIN).astype(float)
results={}; print(f"{'cut':8s} {'B':>7s} {'chi2':>9s} {'dof':>4s} {'PTE':>9s}")
for tag,lab,q_cat in CUTS:
    Dl_full=np.asarray(predict_dl(q_cat,bestfit[tag]))
    D=np.loadtxt(os.path.join(BP_DIR,f"Dl_yy_{tag}_binned_18.txt"))
    C=np.load(os.path.join(CV_DIR,f"cov_full_{tag}_Dl_yy_binned_18.npy"))
    T_b=np.load(os.path.join(CV_DIR,f"T_binned_{tag}_Dl_yy_18.npy"))
    sl=slice(18-KEEP,18); ell=D[sl,0]; Dl_obs=D[sl,1]; Cs=C[sl,sl]; Dl_th=Dl_full[sl]
    err=np.sqrt(np.diag(Cs)); r=Dl_obs-Dl_th; chi2=float(r@np.linalg.inv(Cs)@r); dof=KEEP-1
    # theory error bars: Knox Gaussian at best-fit D_l + trispectrum diagonal
    fsky_eff=FSKY[tag]
    delta_ell=delta_ell_all[sl]
    sigma_g=np.sqrt(2.0*Dl_th**2/((2.0*ell+1.0)*delta_ell*fsky_eff))
    sigma_t=np.sqrt(np.diag(T_b)[sl]/(4.0*np.pi*fsky_eff))
    sigma_th=np.sqrt(sigma_g**2+sigma_t**2)
    results[tag]=dict(ell=ell,Dl_obs=Dl_obs,err=err,Dl_th=Dl_th,
                      sigma_g=sigma_g,sigma_t=sigma_t,sigma_th=sigma_th,
                      chi2=chi2,dof=dof)
    print(f"{lab:8s} {bestfit[tag]:7.4f} {chi2:9.2f} {dof:4d} {_chi2.sf(chi2,dof):9.2e}  "
          f"T/G@ell~{ell[-1]:.0f}={sigma_t[-1]/sigma_g[-1]:.1f}")

I0000 00:00:1782461595.385421 2719045 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782461595.385752 2719045 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1782461596.888038 2719045 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782461596.888481 2719045 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


cut            B      chi2  dof       PTE


Full sky  1.1619      2.17    8  9.75e-01  T/G@ell~960=19.8


q > 5     1.0656   1159.42    8 5.60e-245  T/G@ell~960=2.8


q > 10    1.1006    414.57    8  1.43e-84  T/G@ell~960=4.6


q > 20    1.1244    219.27    8  5.49e-43  T/G@ell~960=7.4


q > 50    1.1506     47.79    8  1.08e-07  T/G@ell~960=12.9


## Best-fit PS vs bandpowers (ell>100)

In [5]:
from matplotlib.lines import Line2D
fig,(ax,axr)=plt.subplots(2,1,figsize=(7.2,7.0),sharex=True,gridspec_kw=dict(height_ratios=[3,1],hspace=0.07))
for (tag,lab,_),col in zip(CUTS,colors):
    r=results[tag]
    ax.plot(r["ell"],r["Dl_th"],"-",color=col,lw=2,zorder=3)
    ax.fill_between(r["ell"], r["Dl_th"]-r["sigma_g"], r["Dl_th"]+r["sigma_g"],
                    color=col, alpha=0.18, zorder=2)
    ax.fill_between(r["ell"], r["Dl_th"]-r["sigma_th"], r["Dl_th"]+r["sigma_th"],
                    color=col, alpha=0.10, zorder=1)
    ax.errorbar(r["ell"],r["Dl_obs"],yerr=r["err"],fmt="o",color=col,ms=4,capsize=2,elinewidth=1,mfc="white",zorder=4)
    axr.errorbar(r["ell"],(r["Dl_obs"]-r["Dl_th"])/r["err"],yerr=1.0,fmt="o",color=col,ms=3,capsize=2,elinewidth=1,mfc="white")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_ylabel(r"$D_\ell^{yy}=\ell(\ell+1)C_\ell^{yy}/2\pi$")
ax.set_title(r"Best-fit tSZ PS (arnaudB1, $\ell>100$; A10 GNFW, $\sigma_{\ln Y}=0.173$)")
ax.grid(True,which="both",alpha=0.2)
handles=[Line2D([0],[0],color=c,marker="o",mfc="white",lw=2,label=f"{lab} (B={bestfit[tag]:.3f})") for (tag,lab,_),c in zip(CUTS,colors)]
handles+=[
    Line2D([0],[0],color="k",lw=2,label="hmfast best fit"),
    Line2D([0],[0],color="k",alpha=0.35,lw=6,label="theory ±1σ Gaussian"),
    Line2D([0],[0],color="k",alpha=0.18,lw=6,label="theory ±1σ Gaussian + trispectrum"),
    Line2D([0],[0],color="k",marker="o",mfc="white",ls="none",label="bandpowers"),
]
ax.legend(handles=handles,fontsize=7.5,loc="upper left",framealpha=0.9)
axr.axhline(0,color="k",lw=0.8); axr.set_ylabel(r"$(D^{\rm obs}-D^{\rm th})/\sigma$"); axr.set_xlabel(r"multipole $\ell$")
axr.grid(True,which="both",alpha=0.2)
fig.savefig(os.path.join(OUTDIR,"bestfit_tsz_ps_arnaudB1_ellgt100.png"),dpi=300,bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR,"bestfit_tsz_ps_arnaudB1_ellgt100.pdf"),bbox_inches="tight")
print("saved ->",OUTDIR); plt.show()

saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb25_arnaudB1_fitB_ellgt100


## Notes

- Selection (which clusters define the masked maps) uses the A10 GNFW B=1 SNR; the
  *signal* bias B is the only free parameter, fixed cosmology + sigma_lnY=0.173.
- Full sky is fit well (chi2/dof ~ 0.3, B ~ 1.16). The masked cuts retain large chi2
  under the tight Gaussian+trispectrum covariance: the smooth completeness only
  approximates the 5 R500c disc masking of the real detected clusters.
- B rises with the q-cut, consistent with mass-dependent bias (more massive,
  higher-q clusters carry larger B). dof = 9 - 1.